In [ ]:
!pip install ultralytics

from ultralytics import YOLO
import json

# 1. 모델 공통 설정
model_path = "/workspace/runs/train/yolo11_climbing_seg/weights/best.pt"
data_yaml_path="/workspace/hold_seg_data-1/data.yaml"

model = YOLO(model_path)


# ==========================================
# [모듈 1] 모델 검증 (Val) - 성능 지표 추출용
# ==========================================
def run_validation(data_yaml_path, device=0):
    print("--- 모델 성능 검증 (Val) 시작 ---")
    
    # val 데이터셋을 이용해 성능 평가 수행
    metrics = model.val(data=data_yaml_path, split="val", device=device)
    # metrics 객체 데이터 구조:
    # - metrics.box (바운딩 박스 지표)
    #   - .map: float (mAP50-95)
    #   - .map50: float (mAP50)
    #   - .p: list/ndarray (Precision 평균)
    #   - .r: list/ndarray (Recall 평균)
    # - metrics.seg (세그멘테이션 마스크 지표)
    #   - .map: float (Mask mAP50-95)
    #   - .map50: float (Mask mAP50)

    # 주요 성능 지표 수치화
    performance_summary = {
        "box_map50_95": float(metrics.box.map),       # Box mAP 0.5:0.95
        "box_map50": float(metrics.box.map50),         # Box mAP 0.5
        "seg_map50_95": float(metrics.seg.map),       # Mask mAP 0.5:0.95
        "seg_map50": float(metrics.seg.map50),         # Mask mAP 0.5
        "precision": float(metrics.box.p.mean()),      # 정밀도 평균
        "recall": float(metrics.box.r.mean())          # 재현율 평균
    }
    
    print(f"검증 완료 - Mask mAP50-95: {performance_summary['seg_map50_95']:.4f}")
    
    # 성능 지표를 JSON 파일로 저장
    with open("yolo_validation_metrics.json", "w", encoding="utf-8") as f:
        json.dump(performance_summary, f, ensure_ascii=False, indent=4)
        
    return performance_summary


# ==========================================
# [모듈 2] 다중 이미지 추론 및 테스트 (Predict)
# ==========================================
def run_prediction(test_source, device="cpu"):
    print("\n--- 다중 이미지 추론 (Predict) 시작 ---")
    
    results = model.predict(
        source=test_source,
        conf=0.5,
        imgsz=640,
        save=True,       # 결과 이미지를 runs/segment/predict 폴더에 자동 저장
        save_txt=False,
        project="/workspace/runs/test_label",  # 저장할 상위 폴더 이름
        name="predicted_seg_image",         # 하위 폴더 이름
        device=device
    )
    
    # results 객체 데이터 구조:
    # - results: list (입력된 이미지 수만큼의 Results 객체 리스트)
    #   - result.path: str (처리된 이미지 파일 경로)
    #   - result.names: dict (클래스 ID와 이름 매핑, 예: {0: 'cat', 1: 'dog'})
    #   - result.boxes: Boxes 객체 (바운딩 박스 관련 정보)
    #     - .cls: tensor (탐지된 객체의 클래스 ID 리스트)
    #     - .conf: tensor (탐지된 객체의 신뢰도 점수 리스트)
    #     - .xyxy: tensor (바운딩 박스 좌표 [x1, y1, x2, y2])
    #   - result.masks: Masks 객체 (세그멘테이션 마스크 정보)
    #     - .xy: list (각 객체의 폴리곤 외곽선 좌표 리스트)

    all_images_data = []

    for r_idx, result in enumerate(results):
        boxes = result.boxes
        masks = result.masks
        image_path = result.path
        
        img_result = {
            "image_index": r_idx,
            "image_path": image_path,
            "detected_objects": []
        }
        
        for i in range(len(boxes)):
            class_id = int(boxes.cls[i])          # 탐지된 클래스 ID
            class_name = model.names[class_id]    # 클래스 이름 문자열
            confidence = float(boxes.conf[i])     # 신뢰도 점수 (0.0 ~ 1.0)
            xyxy = boxes.xyxy[i].tolist()         # 바운딩 박스 좌표 [x1, y1, x2, y2]
            
            # 마스크 폴리곤 좌표 추출 (마스크가 존재하는 경우에만 처리)
            polygon = masks.xy[i].tolist() if masks is not None else None
            
            obj_info = {
                "class_name": class_name,
                "confidence": round(confidence, 4),
                "bounding_box": [round(c, 2) for c in xyxy],
                "polygon": polygon
            }
            img_result["detected_objects"].append(obj_info)
            
        all_images_data.append(img_result)
        print(f"[{image_path}] - 탐지된 객체 수: {len(boxes)}개")

    # 추론 결과를 JSON 파일로 저장
    with open("/workspace/runs/test_label/multi_image_inference_results.json", "w", encoding="utf-8") as f:
        json.dump(all_images_data, f, ensure_ascii=False, indent=4)
        
    return all_images_data


if __name__ == "__main__":
    # CPU 환경에서 실행할 경우 device="cpu" 지정
    # 1. 성능 지표 추출 실행
    #run_validation(data_yaml_path, device=0)
    
    # 2. 다중 이미지 추론 실행
    # run_prediction(test_source="path/to/test_images_folder", device=0)
    print("end")